# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
#your code here
#your code here
def describe_clean_df(df):
    print(f'df shape: {df.shape}')
    print('====================')
    print(f'df data types: {df.dtypes}')
    print('====================')
    print(f'df nulls?:\n {df.isnull().sum()}')
    print('====================')
    
    # 1. Explicitly create a deep copy to break the link to the original df
    df_clean = df.dropna().copy()
    
    # 2. Use bracket notation for assignment
    df_clean['Cabin'] = df_clean['Cabin'].str.split('/').str[0]
    
    # 3. Make sure you catch this returned dataframe when you call the function!
    return df_clean

In [5]:
X=describe_clean_df(spaceship)

df shape: (8693, 14)
df data types: PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object
df nulls?:
 PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [6]:
def features_target_extract(df):
    # 1. Drop the unnecessary ID and Name columns
    X = df.drop(columns=drop_cols, errors='ignore')
    
    # 2. Check if the target column exists (Training vs. Test set handling)
    if target_col in X.columns:
        features = X.drop(columns=[target_col])
        target = X[target_col]
        return X,features, target
    else:
        # If it's the test set, there is no target to drop or return
        features = X
        return X,features

In [7]:
drop_cols = ['PassengerId', 'Name']
target_col = 'Transported'
X,features,target =features_target_extract(X)

In [8]:
from sklearn.preprocessing import OneHotEncoder
    
nonnum_cols = X.select_dtypes(include='object').columns
    
X_train,X_test, y_train,y_test = train_test_split(features,target,test_size=0.2, random_state=0)
    
    #dummies for non-numerical columns, for feature train data
ohe = OneHotEncoder(sparse_output=False, drop='first') # To avoid having an sparse_matrix as output
ohe.fit(X_train[nonnum_cols]) # The .fit() method determines the unique values of each column
X_train_trans_np = ohe.transform(X_train[nonnum_cols])
X_train_trans_df = pd.DataFrame(X_train_trans_np, columns = ohe.get_feature_names_out(nonnum_cols), index=X_train.index)

    #dummies for non-numerical columns, for feature test data
X_test_trans_np = ohe.transform(X_test[nonnum_cols])
X_test_trans_df=pd.DataFrame(X_test_trans_np, columns=ohe.get_feature_names_out(nonnum_cols), index=X_test.index)

    #Combining data again
X_train_num= X_train.select_dtypes('number')
X_test_num = X_test.select_dtypes('number')
    
X_train_full = pd.concat([X_train_trans_df, X_train_num], axis=1)
X_test_full  = pd.concat([X_test_trans_df, X_test_num], axis=1)

In [9]:
    #scaling all dataset, num columns so it is comparable
from sklearn.preprocessing import MinMaxScaler, StandardScaler
    
std_scaler = StandardScaler()
std_scaler.fit(X_train_full)
    
X_train_full_np = std_scaler.transform(X_train_full)
X_test_full_np = std_scaler.transform(X_test_full)
    
X_train_full_np_df = pd.DataFrame(X_train_full_np, columns=X_train_full.columns, index=X_train_full.index)
X_test_full_np_df  = pd.DataFrame(X_test_full_np, columns=X_test_full.columns, index=X_test_full.index)

In [11]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [26]:
#your code here
#your code here
reg = DecisionTreeRegressor(max_depth=20)

reg.fit(X_train_full_np_df, y_train) #standarized dataset X_train
y_pred_test_dt = reg.predict(X_test_full_np_df)

- Evaluate your model

In [27]:
#your code here
print(f"MAE {mean_absolute_error(y_pred_test_dt, y_test): .2f}")
print(f"MSE {mean_squared_error(y_pred_test_dt, y_test): .2f}")
print(f"RMSE {root_mean_squared_error(y_pred_test_dt, y_test): .2f}")
print(f"R2 score {reg.score(X_test_full_np_df, y_test): .2f}")

MAE  0.26
MSE  0.21
RMSE  0.46
R2 score  0.17


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [28]:
import optuna
import optuna.visualization as vis
import time

import scipy.stats as st

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error, make_scorer

from sklearn.model_selection import cross_val_score

In [29]:

parameter_grid = {"max_depth": [10, 50],
                  "min_samples_split": [4, 16],
                  "max_leaf_nodes": [250, 100],
                  "max_features": ["sqrt", "log2"]} # In example we're going to test 2 * 2 * 2 * 2 = 16 combinations of hyperparameters

- Run Grid Search

In [32]:
#your code here
# First we need to setup a dicstionary with all the values that we want to try for each hyperparameter


# We create an instance or our machine learning model
dt = DecisionTreeRegressor(random_state=123)

# We need to set this two variables to be able to compute a confidence interval
confidence_level = 0.95 # confidence_level = 1 - alpha
folds = 10

# Now we need to create an intance of the GridSearchCV class
# The option "scoring", alongside with new_metric ="make_scorer" allows you to evaluate the model performance using a different error metrics.
gs = GridSearchCV(dt, param_grid=parameter_grid, cv=folds, verbose=10) # Here the "cv" allows you to define the number of folds to use.

start_time = time.time()
gs.fit(X_train_full_np_df, y_train)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")


print(f"The best combination of hyperparameters has been: {gs.best_params_}")
print(f"The R2 is: {gs.best_score_: .4f}")

results_gs_df = pd.DataFrame(gs.cv_results_).sort_values(by="mean_test_score", ascending=False)

gs_mean_score = results_gs_df.iloc[0,-3]
gs_sem = results_gs_df.iloc[0,-2] / np.sqrt(folds) # ( std / sqrt(folds))

# Getting the critical value.
gs_tc = st.t.ppf(1-((1-confidence_level)/2), df=folds-1)
gs_lower_bound = gs_mean_score - ( gs_tc * gs_sem )
gs_upper_bound = gs_mean_score + ( gs_tc * gs_sem )

print(f"The R2 confidence interval for the best combination of hyperparameters is: [{gs_lower_bound: .4f}, {gs_upper_bound: .4f}] ")

# Let's store the best model
best_model = gs.best_estimator_



Fitting 10 folds for each of 16 candidates, totalling 160 fits
[CV 1/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 1/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.285 total time=   0.0s
[CV 2/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 2/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.416 total time=   0.0s
[CV 3/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 3/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.390 total time=   0.0s
[CV 4/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 4/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.318 total time=   0.0s
[CV 5/10; 1/16] START max_depth=10, max_features=sqrt

- Evaluate your model

In [33]:
# Now is time evaluate the model in the test set
y_pred_train_df = best_model.predict(X_train_full_np_df)
y_pred_test_df  = best_model.predict(X_test_full_np_df)

print("\n")
print(f"Test MAE: {mean_absolute_error(y_pred_test_df, y_test): .4f}")
print(f"Test MSE: {mean_squared_error(y_pred_test_df, y_test): .4f}")
print(f"Test RMSE: {root_mean_squared_error(y_pred_test_df, y_test): .4f}")
print(f"Test R2 score:  {best_model.score(X_test_full_np_df, y_test): .4f}")
print("\n")




Test MAE:  0.3149
Test MSE:  0.1739
Test RMSE:  0.4170
Test R2 score:   0.3044


